# Q1: Student MDP – Value Functions and Q-Values

**Silver Lecture 2, pages 29–47**

This notebook demonstrates:
1. Policy evaluation under uniform random policy → V^π and Q^π
2. Value iteration → V* and Q*
3. Optimal policy extraction

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

from src.q1_student_mdp.mdp import (
    ACTIONS,
    S_IDX,
    STATES,
    _uniform_policy,
    compute_q_from_v,
    optimal_policy,
    optimal_q,
    policy_evaluation,
    print_policy,
    print_q,
    print_values,
    value_iteration,
)

## MDP Structure

In [ ]:
print('States:', STATES)
print()
print('Actions per state:')
for s, name in enumerate(STATES):
    print(f'  {name}: {ACTIONS[s]}')

## Step 1: Policy Evaluation (V^π and Q^π)

Using the uniform random policy with γ = 1.0 (undiscounted).

**Bellman Expectation:** $V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)[R + \gamma V^\pi(s')]$

In [ ]:
gamma = 1.0
rand_policy = _uniform_policy()

V_pi = policy_evaluation(rand_policy, gamma=gamma)
print_values(V_pi, label=f'V^π  (uniform policy, γ={gamma})')

In [ ]:
Q_pi = compute_q_from_v(V_pi, gamma=gamma)
print_q(Q_pi, label='Q^π (uniform policy)')

### Consistency check: V^π(s) = Σ_a π(a|s) Q^π(s,a)

In [ ]:
print('Consistency check (max error should be < 1e-6):')
for s, name in enumerate(STATES):
    v_from_q = sum(rand_policy[s][a] * Q_pi[s][a] for a in ACTIONS[s])
    err = abs(v_from_q - V_pi[s])
    print(f'  {name}: V^π={V_pi[s]:.4f}, Σ_a π Q={v_from_q:.4f}, err={err:.2e}')

## Step 2: Value Iteration → V* and Q*

**Bellman Optimality:** $V^*(s) = \max_a \sum_{s'} P(s'|s,a)[R + \gamma V^*(s')]$

In [ ]:
V_star = value_iteration(gamma=gamma)
print_values(V_star, label='V* (optimal, γ=1)')

Q_star = optimal_q(V_star, gamma=gamma)
print_q(Q_star, label='Q* (optimal)')

pi_star = optimal_policy(V_star, gamma=gamma)
print_policy(pi_star)

## Visualisation: V^π vs V*

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(STATES))
width = 0.35

ax1.bar(x - width/2, V_pi, width, label='V^π (uniform)', color='steelblue', alpha=0.8)
ax1.bar(x + width/2, V_star, width, label='V* (optimal)', color='coral', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(STATES)
ax1.set_ylabel('Value')
ax1.set_title('State Values: V^π vs V* (γ=1)')
ax1.legend()
ax1.axhline(0, color='black', linewidth=0.5)
ax1.grid(axis='y', alpha=0.3)

# Q-values for C1
c1 = S_IDX['C1']
actions_c1 = list(Q_pi[c1].keys())
q_pi_c1 = [Q_pi[c1][a] for a in actions_c1]
q_star_c1 = [Q_star[c1][a] for a in actions_c1]

x2 = np.arange(len(actions_c1))
ax2.bar(x2 - width/2, q_pi_c1, width, label='Q^π', color='steelblue', alpha=0.8)
ax2.bar(x2 + width/2, q_star_c1, width, label='Q*', color='coral', alpha=0.8)
ax2.set_xticks(x2)
ax2.set_xticklabels(actions_c1)
ax2.set_ylabel('Q-value')
ax2.set_title('Q-values for State C1')
ax2.legend()
ax2.axhline(0, color='black', linewidth=0.5)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('q1_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q1_values.png')

## Discounted Case (γ = 0.9)

In [ ]:
gamma2 = 0.9
V_pi_09 = policy_evaluation(rand_policy, gamma=gamma2)
V_star_09 = value_iteration(gamma=gamma2)

print_values(V_pi_09, label=f'V^π (γ={gamma2})')
print_values(V_star_09, label=f'V* (γ={gamma2})')